# Export 8bit Game ERP Images for Label Studio

This notebook exports **up to 500 real ERP images** from the new 8bit game dataset under `notebooks/datasets/ds003517`.

Properties:
- only the **game events**: `SHOOT_BUTTON`, `COLLECT_STAR`, `MISSILE_HIT_ENEMY`, `PLAYER_CRASH_WALL`, `PLAYER_CRASH_ENEMY`, `COLLECT_AMMO`
- **no downscaling of the ERP matrix**
- **low-pass remains active**
- `trial_type` is handled separately, but is **not** split into `parts` anymore
- export of PNGs plus `tasks_*.csv` and `tasks_*.json` for Label Studio

Exactly **one** ERP image is generated per `trial_type` and channel. Trials remain in the correct `onset_s` order within each image.


In [ ]:
import Pkg

week15_dir = if isfile(joinpath(pwd(), "export_erp_images_labelstudio.ipynb"))
    pwd()
else
    joinpath(pwd(), "notebooks", "week_15")
end

Pkg.activate(joinpath(week15_dir, "..", "model_test"))

using CSV
using CairoMakie
using DataFrames

include(joinpath(week15_dir, "labelstudio_export_helpers.jl"))
using .Week15LabelStudioExport


In [ ]:
data_bundle = load_8bit_export_data()
trial_types = collect(DEFAULT_GAME_TRIAL_TYPES)
candidates = build_candidate_records(data_bundle; trial_types = trial_types, sort_col = DEFAULT_SORT_COLUMN)
target_count = min(500, length(candidates))
selections = select_balanced_records(candidates; target_count = target_count)
distribution_df = sort_distribution_df(selections)

println("8bit EEG shape: ", data_bundle.full_shape)
println("Events rows: ", size(data_bundle.events, 1))
println("Game trial types: ", trial_types)
println("Available candidate images: ", length(candidates))
println("Planned export count: ", target_count)
println("Export root: ", DEFAULT_EXPORT_ROOT)

distribution_df


In [ ]:
preview_positions = unique(Int.(round.([1, max(1, length(selections) * 0.25), max(1, length(selections) * 0.5), max(1, length(selections) * 0.75), length(selections)])))
preview = build_preview_images(data_bundle, selections; indices = preview_positions)
println("Preview indices: ", preview.indices)
fig_preview = plot_erp_grid(preview.images, preview.metadata; n_cols = 2)
fig_preview


In [ ]:
export_bundle = export_labelstudio_images(; target_count = 500)

println("Available candidates: ", export_bundle.available_count)
println("Exported images: ", export_bundle.exported_count)
println("Images dir: ", export_bundle.images_dir)
println("CSV manifest: ", export_bundle.csv_manifest_path)
println("JSON manifest: ", export_bundle.json_manifest_path)

first(export_bundle.manifest_df, min(10, nrow(export_bundle.manifest_df)))


## Label Studio Note

- The exported files are written to `notebooks/week_15/label_studio_data_unlabelled_week15_8bit_game_500`.
- If the derived 8bit files are missing, they are generated from `notebooks/datasets/ds003517` into `notebooks/datasets/ds003517_sub001_derived`.
- For **Local Files** in Label Studio, `LOCAL_FILES_DOCUMENT_ROOT` should point to `notebooks/week_15`.
- The task file for import is `tasks_unlabelled_week15_8bit_game_500.json`.
